In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("cars.csv")

# Basic information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
print(df.head())

# Numerical summary
print(df.describe())

# Median of numerical columns
print("\nMedian:")
print(df.median(numeric_only=True))

# Standard deviation
print("\nStandard Deviation:")
print(df.std(numeric_only=True))

# Skewness (distribution shape)
print("\nSkewness:")
print(df.skew(numeric_only=True))

# Data cleaning: remove invalid production_year entries
df_clean = df[df["production_year"] > 1900].copy()
print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(df_clean))

# Inspect only the categorical columns relevant to this analysis
print(df["Marka"].value_counts().head(15))
print("\nNumber of unique brands:", df["Marka"].nunique())
print("\n", df["production_year"].value_counts().sort_index())

# Descriptive statistics for groups (Car Brand)
group_stats = df.groupby("Marka")["price_x"].agg(
    Mean="mean",
    Median="median",
    Std="std",
    Skew="skew"
)

print(group_stats)

Shape: (653721, 56)

Columns:
['id_x', 'car_rel_url_x', 'datetime_scrape', 'name', 'price_x', 'currency_x', 'datetime_product', 'city', 'day', 'hour', 'attributes', 'production_year', 'engine_displacement_num', 'engine_displacement_unit', 'kilometrage_num', 'kilometrage_unit', 'barter', 'loan', 'salon', 'spare_parts', 'vip', 'featured', 'img_url', 'id_y', 'cars_id', 'car_rel_url_y', 'datetime', 'description', 'price_y', 'currency_y', 'owner_name', 'shop_name', 'phone', 'updated', 'views', 'vin', 'car_details_id_x', 'Ban növü', 'Buraxılış ili', 'Hansı bazar üçün yığılıb', 'Marka', 'Model', 'Mühərrik', 'Qəzalı', 'Rəng', 'Sahiblər', 'Sürətlər qutusu', 'Vəziyyəti', 'Yeni', 'Yerlərin sayı', 'Yürüş', 'Ötürücü', 'Şəhər', 'car_details_id_y', 'car_rel_url', 'extra_info']

Data Types:
id_x                         object
car_rel_url_x                object
datetime_scrape              object
name                         object
price_x                     float64
currency_x                   objec

## Checkpoint 1: Descriptive Statistics for Groups

The dataset was grouped by car brand (Marka). For each group, the mean, median, standard deviation, and skewness of price_x were calculated. These descriptive statistics summarize the central tendency, variability, and distribution shape of car prices, allowing comparison between different car brands. Note that some brands have very few listings (even n=1), so their std/skew values are NaN or unreliable and should not be over-interpreted.

In [3]:
group_stats = df.groupby("Marka")["price_x"].agg(
    Mean="mean",
    Median="median",
    Std="std",
    Skew="skew"
)

print(group_stats)

                    Mean   Median           Std      Skew
Marka                                                    
ATV          3508.333333   3950.0   1691.276638 -0.814495
Abarth      16694.495413  18100.0   3071.518953 -0.960685
Acura       18007.575758  18000.0   2837.145386 -1.798210
Alfa Romeo  65978.372591  54600.0  31553.669504  0.403725
Aprilia     17914.191919   8399.0   9774.995984  0.069288
...                  ...      ...           ...       ...
ZXMCO        2625.000000   3500.0   1371.039751 -1.064886
Zamyad      38700.000000  38700.0           NaN       NaN
Zongshen     4733.166667   5799.5   2355.325915 -0.711247
Zontes       3828.571429   4200.0   1569.483135 -0.011791
iCar        24900.000000  24900.0      0.000000  0.000000

[208 rows x 4 columns]


## Checkpoint 2: Hypothesis Formulation

**Business Question 1**
Do luxury car brands have a higher average price than non-luxury car brands?

- H0 (Null Hypothesis): There is no significant difference in the average price between luxury and non-luxury car brands.
- H1 (Alternative Hypothesis): Luxury car brands have a significantly different average price than non-luxury car brands.

**Business Question 2**
Does the average car price differ across production years?

- H0 (Null Hypothesis): The mean car price is the same across all production years.
- H1 (Alternative Hypothesis): At least one production year has a different mean car price.

Note: these tests establish association, not causation — other factors (mileage, condition, engine size, etc.) are not controlled for here.

In [4]:
# Business Question 1: Luxury vs Non-luxury cars
luxury_brands = [
    "Mercedes", "BMW", "Audi", "Lexus", "Porsche",
    "Land Rover", "Jaguar", "Bentley", "Maserati", "Ferrari"
]

luxury = df[df["Marka"].isin(luxury_brands)]["price_x"]
non_luxury = df[~df["Marka"].isin(luxury_brands)]["price_x"]

print("Luxury cars:", len(luxury))
print("Non-luxury cars:", len(non_luxury))

# Business Question 2: Price across production years
# using df_clean to exclude invalid production_year == 0 entries
year_groups = [
    group["price_x"].dropna()
    for _, group in df_clean.groupby("production_year")
]

print("Number of production year groups:", len(year_groups))

Luxury cars: 200728
Non-luxury cars: 452993
Number of production year groups: 72


## Checkpoint 3: Selection and Execution of Statistical Tests

Two statistical tests were selected according to the type of business question.

- An independent two-sample **t-test** (Welch's, unequal variances) was used to compare the mean prices of luxury and non-luxury car brands.
- A one-way **ANOVA** was used to determine whether the mean car price differs across production years.

Since price data is heavily right-skewed (see Checkpoint 1) and assumption tests in Checkpoint 5 show non-normality, **non-parametric equivalents** (Mann-Whitney U and Kruskal-Wallis) are also run as a robustness check. With sample sizes in the hundreds of thousands, the Central Limit Theorem means the t-test/ANOVA are still reasonably reliable for comparing means, but the non-parametric tests confirm the conclusion doesn't depend on that assumption.

In [5]:
from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal

# -------------------------------
# Test 1: Independent t-test (parametric)
# -------------------------------
t_stat, p_value = ttest_ind(
    luxury,
    non_luxury,
    equal_var=False,
    nan_policy="omit"
)

print("Independent t-test (Welch's)")
print("t-statistic:", t_stat)
print("p-value:", p_value)

# Robustness check: Mann-Whitney U (non-parametric)
u_stat, u_p = mannwhitneyu(luxury, non_luxury, alternative="two-sided")
print("\nMann-Whitney U test")
print("U-statistic:", u_stat)
print("p-value:", u_p)

# -------------------------------
# Test 2: One-way ANOVA (parametric)
# -------------------------------
anova_stat, anova_p = f_oneway(*year_groups)

print("\nOne-way ANOVA")
print("F-statistic:", anova_stat)
print("p-value:", anova_p)

# Robustness check: Kruskal-Wallis (non-parametric)
kw_stat, kw_p = kruskal(*year_groups)
print("\nKruskal-Wallis test")
print("H-statistic:", kw_stat)
print("p-value:", kw_p)

Independent t-test (Welch's)
t-statistic: 142.3057823838934
p-value: 0.0

Mann-Whitney U test
U-statistic: 55546942544.0
p-value: 0.0

One-way ANOVA
F-statistic: 3560.843565666654
p-value: 0.0

Kruskal-Wallis test
H-statistic: 304356.1099646499
p-value: 0.0


## Checkpoint 4: Interpretation of p-value and Confidence Interval

The independent t-test and one-way ANOVA both produced p-values far below 0.05, indicating strong evidence against both null hypotheses — this is not just "p < 0.05 = significant" but reflects a very large sample size (n > 650,000), which makes even small differences statistically detectable.

The 95% confidence interval for luxury car prices is approximately 40,836.65–41,190.74 AZN, while the interval for non-luxury cars is approximately 27,414.74–27,531.94 AZN. These intervals represent the range in which the true population mean price is likely to fall, given the sample. Since the intervals do not overlap, this supports the conclusion that luxury cars are priced higher on average — not just that the difference is "significant," but that the estimated gap is large and precise: roughly **13,500 AZN** difference between group means.

It's worth noting that with a dataset this large, statistical significance is almost guaranteed for any real difference, however small — so the confidence intervals and effect size (the actual AZN gap) matter more for business decisions than the p-value alone.

In [6]:
from scipy import stats
import numpy as np

# 95% Confidence Interval for luxury cars
ci_luxury = stats.t.interval(
    confidence=0.95,
    df=len(luxury)-1,
    loc=np.mean(luxury),
    scale=stats.sem(luxury)
)

# 95% Confidence Interval for non-luxury cars
ci_non_luxury = stats.t.interval(
    confidence=0.95,
    df=len(non_luxury)-1,
    loc=np.mean(non_luxury),
    scale=stats.sem(non_luxury)
)

print("Luxury cars 95% CI:", ci_luxury)
print("Non-luxury cars 95% CI:", ci_non_luxury)

# Effect size: mean difference in AZN
mean_diff = luxury.mean() - non_luxury.mean()
print("\nMean price difference (luxury - non-luxury):", round(mean_diff, 2), "AZN")

print("\nT-test p-value:", p_value)
print("Mann-Whitney U p-value:", u_p)
print("ANOVA p-value:", anova_p)
print("Kruskal-Wallis p-value:", kw_p)

Luxury cars 95% CI: (np.float64(40836.64548873269), np.float64(41190.73650082532))
Non-luxury cars 95% CI: (np.float64(27414.74175030478), np.float64(27531.93885623881))

Mean price difference (luxury - non-luxury): 13540.35 AZN

T-test p-value: 0.0
Mann-Whitney U p-value: 0.0
ANOVA p-value: 0.0
Kruskal-Wallis p-value: 0.0


## Checkpoint 5: Assumption Testing

Before applying parametric statistical tests, the assumptions of normality and homogeneity of variances were evaluated.

- The **Shapiro-Wilk test** was used to assess normality (on a random sample of 5,000, since Shapiro-Wilk isn't reliable/feasible on very large samples).
- **Levene's test** was used to check equality of variances, both for the luxury vs. non-luxury comparison and across the production-year groups (relevant to the ANOVA in Checkpoint 3).

Results: both luxury and non-luxury price distributions are **strongly non-normal** (Shapiro-Wilk p ≈ 1.6e-71 and 1.0e-67), and variances are **significantly unequal** between the two groups (Levene's p ≈ 0) and across production years. This means the classical assumptions behind the t-test and one-way ANOVA are violated.

Given this:
- The t-test was already run as **Welch's t-test** (`equal_var=False`), which does not assume equal variances — appropriate here.
- The one-way ANOVA does assume equal variances across all groups, which is violated here, so its result should be treated cautiously.
- The **Mann-Whitney U** and **Kruskal-Wallis** tests (Checkpoint 3) don't require normality or equal variances, and both confirm the same conclusions as the parametric tests — this gives confidence the results are robust despite the violated assumptions.

In [7]:
from scipy.stats import shapiro, levene

# Shapiro-Wilk normality test (sampled, since Shapiro-Wilk is not valid/feasible on huge n)
shapiro_luxury = shapiro(luxury.sample(5000, random_state=42))
shapiro_non_luxury = shapiro(non_luxury.sample(5000, random_state=42))

print("Shapiro-Wilk Test")
print("Luxury cars:", shapiro_luxury)
print("Non-luxury cars:", shapiro_non_luxury)

# Levene's test: luxury vs non-luxury
levene_test = levene(luxury, non_luxury)
print("\nLevene's Test (luxury vs non-luxury)")
print("Statistic:", levene_test.statistic)
print("p-value:", levene_test.pvalue)

# Levene's test: across production_year groups (relevant to the ANOVA)
levene_years = levene(*year_groups)
print("\nLevene's Test (across production years)")
print("Statistic:", levene_years.statistic)
print("p-value:", levene_years.pvalue)

Shapiro-Wilk Test
Luxury cars: ShapiroResult(statistic=np.float64(0.6683507508110935), pvalue=np.float64(1.5940140517330788e-71))
Non-luxury cars: ShapiroResult(statistic=np.float64(0.7255909026469335), pvalue=np.float64(1.0316493954570618e-67))

Levene's Test (luxury vs non-luxury)
Statistic: 25426.24559970701
p-value: 0.0

Levene's Test (across production years)
Statistic: 1555.428181651275
p-value: 0.0


## Checkpoint 6: Business Conclusion

The statistical analysis showed that luxury cars are associated with significantly higher average prices than non-luxury cars (a gap of roughly 13,500 AZN), and that average car prices also vary significantly across production years. Both the t-test/ANOVA and their non-parametric counterparts (Mann-Whitney U, Kruskal-Wallis) agree on these conclusions, so the results are robust to the non-normality found in the data.

These results indicate that brand category and production year are strongly **associated with** price — not that they *cause* price differences on their own. This is observational marketplace data, not a controlled experiment: other factors not accounted for here (mileage, condition, engine size, market origin, etc.) likely also drive both price and brand/year patterns, so causal claims aren't supported by this analysis alone.

Note on scope: the one-way ANOVA tests whether price differs *somewhere* across the 72 production-year groups, which it does — but it does not identify *which* specific years differ from which. Answering that would require pairwise post-hoc comparisons (e.g., Tukey's HSD) with a multiple-comparisons correction, since testing many pairs without correction inflates the false-positive rate. That is left as a natural next step beyond this analysis.

From a business perspective, these findings can still support practical decisions: pricing strategy can account for the brand premium luxury vehicles command, and inventory/market segmentation can be informed by how prices trend across production years — while recognizing that a deeper causal or predictive model (e.g., regression controlling for mileage and condition) would be needed before making stronger pricing decisions.